# Inference steps of COMPASS-PTM

In [1]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, roc_auc_score, average_precision_score, matthews_corrcoef, roc_curve
import warnings
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.preprocessing import label_binarize
from config_pep import config
from dataset import load_data_, load_data_finetune_binary, load_data_inference, load_data_finetune_inference
from model import create_model_binary, create_model_trans_bias
from transformers import AutoTokenizer

/data0/lsn1/miniforge3/envs/compass-ptm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocess_sequence(sequence, window_size=50):
    pep_seq_list = []
    sequence = sequence.upper()
    i = 0
    while i < len(sequence):
        if i + window_size <= len(sequence):
            pep_seq = sequence[i:i + window_size]
        else:
            pep_seq = sequence[i:]
            padding = window_size - len(pep_seq)
            print(len(pep_seq), padding)
            pep_seq = sequence[i:] + 'X' * padding
        # add flanking regions to the peptide sequence, if available
        start = max(0, i - 10)
        end = min(len(sequence), i + window_size + 10)
        pep_seq_extend = sequence[start:end]
        if start == 0:
            pep_seq_extend = 'X' * (10 - i) + pep_seq_extend
        if end == len(sequence):
            pep_seq_extend = pep_seq_extend + 'X' * (10 - (len(sequence) - i - window_size))
        pep_seq_list.append(pep_seq_extend)
        i += window_size
    return pep_seq_list

In [5]:
def load_model(ckpt_path):
    model = create_model_trans_bias(mode='lora', model_checkpoint=config.model_checkpoint).to(config.device) # full, lora, last_layer
    
    mlp = None
    state_dict = torch.load(ckpt_path, map_location=config.device)
    model.load_state_dict(state_dict)
    
    return model, mlp

In [14]:
def inference(model, mlp, data_loader, sequence, device, threshold=0.5):
    model.eval()
    all_preds = []
    all_probs = []
    seq_length = len(sequence)
    with torch.no_grad():
        for token, _, mask, seq_lens, seq in tqdm(data_loader):
            token, mask, seq_len = token.to(device), mask.to(device), seq_lens.to(device)
            outputs = model(token, mask, seq)

            dim_1, dim_3, dim_4 = outputs.shape
            outputs = outputs.view(dim_1 * dim_3, dim_4)
            probs = torch.sigmoid(outputs)
            all_probs.append(probs.cpu().numpy())
            preds = torch.sigmoid(outputs) > threshold
            all_preds.append(preds.cpu().numpy())
        
        # Concatenate all predictions and probabilities
        all_preds = np.concatenate(all_preds, axis=0)
        print(f"Shape of all_preds: {all_preds.shape}")
        all_preds = all_preds[:seq_length]
        all_probs = np.concatenate(all_probs, axis=0)[:seq_length]

        ptm_types = ['Acetylation', 'Methylation', 'Phosphorylation', 'Succinylation', 'Ubiquitination', 
                     'O-linked Glycosylation', 'N-linked Glycosylation', 'Other']
        ptm_probs = {ptm: [] for ptm in ptm_types}
        for i in range(all_probs.shape[0]):
            for j, ptm in enumerate(ptm_types):
                ptm_probs[ptm].append(all_probs[i, j])

        df = pd.DataFrame(ptm_probs)
        df.to_csv('ptm_probabilities.csv', index=False)
        print("Predictions and probabilities saved to ptm_probabilities.csv")
        
        for i in range(all_preds.shape[0]):
            if all_preds[i].any():
                print(f"Peptide {sequence[i]}{i+1}: Predicted PTMs: ", end="")
                for j, ptm in enumerate(ptm_types):
                    if all_preds[i, j]:
                        print(f"{ptm} (Probability: {all_probs[i, j]:.4f})", end=", ")
                print()

    return all_preds, all_probs

In [ ]:
def load_model_stage2(ckpt_path):
    model = create_model_binary(mode='lora', checkpoint=config.checkpoint, model_checkpoint=config.model_checkpoint).to(config.device)
    mlp = None
    state_dict = torch.load(ckpt_path, map_location=config.device)
    model.load_state_dict(state_dict, strict=False)
    return model, mlp

In [ ]:
def inference_stage2(model, mlp, data_loader, device, threshold=0.5):
    model.eval()
    with torch.no_grad():
        token, labels, mask, seq_lens, kinase, sequences = next(iter(data_loader))
        token, mask, seq_lens, kinase = token.to(device), mask.to(device), seq_lens.to(device), kinase.to(device)
        outputs = model(token, mask, kinase, sequences)
        probs = torch.sigmoid(outputs)
        if probs >= threshold:
            print(f"Predicted: Modified site with probability {probs.item():.4f}")
        else:
            print(f"Predicted: Non-modified site with probability {probs.item():.4f}")

## An Example: P30260

### Stage1: Given a full-length protein sequence, predict all its PTM sites.

In [ ]:
sequence='MTVLQEPVQAAIWQALNHYAYRDAVFLAERLYAEVHSEEALFLLATCYYRSGKAYKAYRLLKGHSCTTPQCKYLLAKCCVDLSKLAEGEQILSGGVFNKQKSHDDIVTEFGDSACFTLSLLGHVYCKTDRLAKGSECYQKSLSLNPFLWSPFESLCEIGEKPDPDQTFKFTSLQNFSNCLPNSCTTQVPNHSLSHRQPETVLTETPQDTIELNRLNLESSNSKYSLNTDSSVSYIDSAVISPDTVPLGTGTSILSKQVQNKPKTGRSLLGGPAALSPLTPSFGILPLETPSPGDGSYLQNYTNTPPVIDVPSTGAPSKKSVARIGQTGTKSVFSQSGNSREVTPILAQTQSSGPQTSTTPQVLSPTITSPPNALPRRSSRLFTSDSSTTKENSKKLKMKFPPKIPNRKTKSKTNKGGITQPNINDSLEITKLDSSIISEGKISTITPQIQAFNLQKAAAEGLMSLLREMGKGYLALCSYNCKEAINILSHLPSHHYNTGWVLCQIGRAYFELSEYMQAERIFSEVRRIENYRVEGMEIYSTTLWHLQKDVALSVLSKDLTDMDKNSPEAWCAAGNCFSLQREHDIAIKFFQRAIQVDPNYAYAYTLLGHEFVLTEELDKALACFRNAIRVNPRHYNAWYGLGMIYYKQEKFSLAEMHFQKALDINPQSSVLLCHIGVVQHALKKSEKALDTLNKAIVIDPKNPLCKFHRASVLFANEKYKSALQELEELKQIVPKESLVYFLIGKVYKKLGQTHLALMNFSWAMDLDPKGANNQIKEAIDKRYLPDDEEPITQEEQIMGTDESQESSMTDADDTQLHAAESDEF'
pep_seq_list = preprocess_sequence(sequence)
print(f"Processed {len(pep_seq_list)} 50mer peptides from the sequence.")
print("Loading pretrained checkpoints")
ckpt_path = '../checkpoint/best_stage1.pth'
model, mlp = load_model(ckpt_path)

print("Loading inferencing dataset")
tokenizer = AutoTokenizer.from_pretrained(config.model_checkpoint)
test_data = load_data_inference(sequence, tokenizer)
test_loader = DataLoader(test_data, batch_size=config.batch_size)

print("Start inferencing...")
preds, probs = inference(model, mlp, test_loader, sequence, config.device, threshold=0.5)

24 26
Processed 17 50mer peptides from the sequence.
Loading pretrained checkpoints


Some weights of EsmForTokenClassification were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/data0/lsn1/jjzhang/COMPASS-PTM/models/model.py:321: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.a

Loading inferencing dataset
['XXXXXXXXXXMTVLQEPVQAAIWQALNHYAYRDAVFLAERLYAEVHSEEALFLLATCYYRSGKAYKAYRL', 'LFLLATCYYRSGKAYKAYRLLKGHSCTTPQCKYLLAKCCVDLSKLAEGEQILSGGVFNKQKSHDDIVTEF', 'ILSGGVFNKQKSHDDIVTEFGDSACFTLSLLGHVYCKTDRLAKGSECYQKSLSLNPFLWSPFESLCEIGE', 'SLSLNPFLWSPFESLCEIGEKPDPDQTFKFTSLQNFSNCLPNSCTTQVPNHSLSHRQPETVLTETPQDTI', 'HSLSHRQPETVLTETPQDTIELNRLNLESSNSKYSLNTDSSVSYIDSAVISPDTVPLGTGTSILSKQVQN', 'SPDTVPLGTGTSILSKQVQNKPKTGRSLLGGPAALSPLTPSFGILPLETPSPGDGSYLQNYTNTPPVIDV', 'SPGDGSYLQNYTNTPPVIDVPSTGAPSKKSVARIGQTGTKSVFSQSGNSREVTPILAQTQSSGPQTSTTP', 'EVTPILAQTQSSGPQTSTTPQVLSPTITSPPNALPRRSSRLFTSDSSTTKENSKKLKMKFPPKIPNRKTK', 'ENSKKLKMKFPPKIPNRKTKSKTNKGGITQPNINDSLEITKLDSSIISEGKISTITPQIQAFNLQKAAAE', 'KISTITPQIQAFNLQKAAAEGLMSLLREMGKGYLALCSYNCKEAINILSHLPSHHYNTGWVLCQIGRAYF', 'LPSHHYNTGWVLCQIGRAYFELSEYMQAERIFSEVRRIENYRVEGMEIYSTTLWHLQKDVALSVLSKDLT', 'TTLWHLQKDVALSVLSKDLTDMDKNSPEAWCAAGNCFSLQREHDIAIKFFQRAIQVDPNYAYAYTLLGHE', 'QRAIQVDPNYAYAYTLLGHEFVLTEELDKALACFRNAIRVNPRHYNAWYGLGMIYYKQEKFSLAEMHFQK', 'LGMIYYKQ

100%|██████████| 1/1 [00:00<00:00, 13.05it/s]

Shape of all_preds: (850, 8)
Predictions and probabilities saved to ptm_probabilities.csv
Peptide K62: Predicted PTMs: Ubiquitination (Probability: 0.7629), 
Peptide K72: Predicted PTMs: Ubiquitination (Probability: 0.5665), 
Peptide K77: Predicted PTMs: Ubiquitination (Probability: 0.7391), 
Peptide K84: Predicted PTMs: Ubiquitination (Probability: 0.7364), 
Peptide K99: Predicted PTMs: Ubiquitination (Probability: 0.9326), 
Peptide S143: Predicted PTMs: Phosphorylation (Probability: 0.5123), 
Peptide S150: Predicted PTMs: Phosphorylation (Probability: 0.9956), 
Peptide S194: Predicted PTMs: Phosphorylation (Probability: 0.5950), 
Peptide T200: Predicted PTMs: Phosphorylation (Probability: 0.7779), O-linked Glycosylation (Probability: 0.9486), 
Peptide N300: Predicted PTMs: N-linked Glycosylation (Probability: 0.9986), 
Peptide K390: Predicted PTMs: Ubiquitination (Probability: 0.6075), 
Peptide K399: Predicted PTMs: Ubiquitination (Probability: 0.5559), 
Peptide K431: Predicted PTMs:

### Stage2: For the PTM site of interest predicted in the first stage, predict whether the enzyme of interest catalyzes the occurrence of the site.
For example:

Site: T446

Enzyme: Q16539

In [6]:
# MSQERPTFYRQELNKTIWEVPERYQNLSPVGSGAYGSVCAAFDTKTGLRVAVKKLSRPFQSIIHAKRTYRELRLLKHMKHENVIGLLDVFTPARSLEEFNDVYLVTHLMGADLNNIVKCQKLTDDHVQFLIYQILRGLKYIHSADIIHRDLKPSNLAVNEDCELKILDFGLARHTDDEMTGYVATRWYRAPEIMLNWMHYNQTVDIWSVGCIMAELLTGRTLFPGTDHIDQLKLILRLVGTPGAELLKKISSESARNYIQSLTQMPKMNFANVFIGANPLAVDLLEKMLVLDSDKRITAAQALAHAYFAQYHDPDDEPVADPYDQSFESRDLLIDEWKSLTYDEVISFVPPPLDQEEMES
ckpt_path = '../checkpoint/best_stage2.pth'
model, mlp = load_model_stage2(ckpt_path)
print("Loading inferencing dataset")
tokenizer = AutoTokenizer.from_pretrained(config.model_checkpoint)
test_data = load_data_finetune_inference('EGKISTITPQIQAFN', 'Q16539', tokenizer)
test_loader = DataLoader(test_data, batch_size=config.batch_size)
    
print("Start inferencing...")
inference_stage2(model, mlp, test_loader, config.device)

Some weights of EsmForTokenClassification were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/data0/lsn1/jjzhang/COMPASS-PTM/models/model.py:311: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.a

Loading inferencing dataset
Max peptide length: 15
Start inferencing...


/data0/lsn1/jjzhang/COMPASS-PTM/models/dataset.py:88: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return input_ids, torch.tensor(label, dtype=torch.float), attention_mask, seq_len, kinase, sequence


Predicted: Modified site with probability 0.6289
